In [ ]:
import numpy as np
import datetime
import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchmetrics
import kornia.augmentation as K

In [ ]:
PROJECT_ROOT = Path().resolve().parent
datetime_now = datetime.now()
date = datetime_now.strftime("%Y-%m-%d")

loss_curve_fp = PROJECT_ROOT / "outputs/plots" / f"loss_curve_{date}.png"

In [ ]:
# def load_patch_dict_as_tensor(Patch_dict):
    
#     # try first as a for loop and then see if can switch to comprehension

#     band_names = ["B2", "B3", "B4", "B5", "B6"]

#     feature_list = Patch_dict["features"]

#     patch_list = []0
#     label_list = []
#     location_list = []

#     for feat in feature_list:

#         band_arrays = [np.array(feat["properties"][band]) for band in band_names]
#         stacked_patch = np.stack(band_arrays) # (C,H,W)

#         patch_list.append(stacked_patch)
#         label_list.append(feat["properties"]["lc"])
#         location_list.append(feat["properties"]["location"])

#     patches = np.stack(patch_list) # (N, C, H, W)
#     patches = torch.from_numpy(patches).float()

#     labels= torch.tensor(label_list).long()
#     locations = np.array(location_list) # array for easier boolean masking but no need for tensor


#     return patches, labels, locations

In [ ]:
patches_path = PROJECT_ROOT / "outputs/15px_patches" / "combined_15px_patches.geojson"

with open(patches_path, "r") as f:
    patches = json.load(f)


In [ ]:
# Version using comprehension

def load_patch_dict_as_tensor(Patch_dict, band_list = None):
    if not band_list:
        band_list = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

    feature_list = Patch_dict["features"]

    patches = np.stack([
        np.stack([np.array(F["properties"][band]) for band in band_list]) 
        for F in feature_list 
    ])

    labels = torch.tensor([F["properties"]["label"] for F in feature_list]).long()

    locations = np.array([F["properties"]["location"] for F in feature_list]) # array for easier boolean masking but no need for tensor

    patches = torch.from_numpy(patches).float()

    patches = patches.permute(0,3,1,2) # changes stacked numpy (N,H,W,C) to pytorch (N,C,H,W)

    return patches, labels, locations

patches, labels, locations = load_patch_dict_as_tensor(patches, band_list= None)

# check the shape

print(patches.shape)


In [ ]:
# next need to calcualte the mean and std for just the train region, can create a test_mask and val_mask and then train_masl = ~test_mask & ~val_mask

test_region = "Valsequillo"
val_region = "Vembanad"

test_mask = locations == test_region
val_mask = locations == val_region

train_mask = ~test_mask & ~ val_mask

train_patches, train_labels = patches[train_mask], labels[train_mask]
val_patches, val_labels = patches[val_mask], labels[val_mask]
test_patches, test_labels = patches[test_mask], labels[test_mask]


mean = train_patches.mean(dim= (0,2,3)) # will average over the 0,2,3 axes and keep the 1 axis (channels / bands) separate. output shape (C,)
std = train_patches.std(dim = (0,2,3))

# mean = torch.tensor(mean).view(1,-1,1,1) # changes mean from (C,) to (1,C,1,1)
# std = torch.tensor(std).view(1,-1,1,1)

# standardise all the patches using the mean and std from the train patches - np maybe do this for each patch in the dataset in __getitem__?

# std_train_patches = (train_patches - mean)/std
# std_val_patches = (val_patches-mean)/std
# std_test_patches = (test_patches-mean)/ std




In [ ]:

class PatchDataset(Dataset):
    def __init__(self, patches, labels, mean, std):
        
        mean = torch.tensor(mean).view(1,-1,1,1) # changes from (C,) to (1,C,1,1)
        std = torch.tensor(std).view(1,-1,1,1)
    
        self.patches = (patches - mean)/std 
        self.labels = labels 

    def __len__(self):
        return len(patches)

    def __getitem__(self, idx):
        return self.patches[idx], self.labels[idx]
    


    
train_dataset = PatchDataset(patches = train_patches, mean = mean, std = std)

train_loader = DataLoader(
    dataset= train_dataset,
    shuffle= True,
    batch_size= 32,
    num_workers= 4,
    pin_memory= True
    )
    
val_dataset = PatchDataset(patches = val_patches, mean = mean, std = std)

val_loader = DataLoader(
    dataset= val_dataset,
    shuffle= True,
    batch_size= 32,
    num_workers= 4,
    pin_memory= True
    )

# # --- Device selection ---
# # Colab (with a GPU runtime: Runtime > Change runtime type > GPU):
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Apple Silicon (M1/M2/M3/M4), using the Metal Performance Shaders backend:
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Portable version that works on either without editing — checks cuda, then mps, then falls back to cpu:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

aug = K.AugmentationSequential([
    K.RandomRotation(degrees = 90.0, p =0.5),
    K.RandomHorizontalFlip(p= 0.25),
    K.RandomVerticalFlip(p= 0.25),
    data_keys= ['input']
]).to(device)

# for epoch in len(number_epochs):
#     # Training
#     #-----------------------
#     model.train()

#     for batch_X, batch_y in train_loader:
#         batch_X = batch_X.to(device)
#         batch_y = batch_y.to(device)

#         batch_X = aug(batch_X) # Augment only in the train split

#         optimizer.zero_grad()
#         output = model(batch_x)
#         loss = criterion(output, batch_y)
#         loss.backwards()
#         optimizer.step()

#     # Val
#     #-----------------------

#     model.eval()

#     with torch.no_grad():
#         for batch_X, batch_y in val_loader:
            
#             batch_X = batch_X.to(device)
#             batch_y = batch_y.to(device)

#             output = model(batch_x)
#             val_loss = criterion(output, batch_y)

    # Plot?

In [ ]:
# Define my model

input_channels = 10


myCNN = nn.Sequential(
    nn.Conv2d(in_channels=input_channels, 
            out_channels = 32, 
            kernel_size=3,
            stride= 1,
            padding=1  ),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.Conv2d(in_channels=32, 
            out_channels = 32, 
            kernel_size=3,
            stride= 1,
            padding=1  ),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
        nn.Conv2d(in_channels=64, 
            out_channels = 64, 
            kernel_size=3,
            stride= 1,
            padding=1  ),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.Conv2d(in_channels=64, 
            out_channels = 64, 
            kernel_size=3,
            stride= 1,
            padding=1  ),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    #nn.Dropout(0.5) wont have this for now
    nn.Linear(64,1)
)

In [ ]:
# metrics - use torchmetrics to make a metric collection that is a dictionary of metrics

num_classes = 2

metrics = torchmetrics.MetricCollection({
    "accuracy": torchmetrics.classification.BinaryAccuracy(),
    "f1": torchmetrics.classification.BinaryF1Score(),
    "precision": torchmetrics.classifcation.BinaryPrecision(),
    "recall": torchmetrics.classification.BinaryRecall()
}).to(device)

In [ ]:
def train_for_one_epoch(model, train_loader, device, aug, optimizer, criterion)
    
    model.train()
    running_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        batch_X = aug(batch_X) # Augment only in the train split

        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backwards()
        optimizer.step()

        running_loss += loss.item()*batch_X.size(0)
        avg_loss = running_val_loss / len(val_loader.dataset) 
        
    return avg_loss

    
def val_for_one_epoch(model, val_loader, metrics, device, criterion)
    
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            output = model(batch_X)
            val_loss = criterion(output, batch_y)
            
            preds = output.argmax(dim = 1)
            metrics.update(preds, batch_y) # accumulates the metrics per batch

            running_val_loss += val_loss.item()*batch_X.size(0)
        
        avg_loss = running_val_loss / len(val_loader.dataset)
        results = {k: v.item() for k, v in metrics.compute().items()}
        
    return avg_loss, results


def plot_and_save_loss(history_dict, current_epoch, loss_curve_fp)
    
    plt.figure(figsize=(8, 5))
    plt.plot(history_dict["train_loss"], label="Train Loss")
    plt.plot(history_dict["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(Path(loss_curve_fp), dpi=150)
    plt.show()

    

In [ ]:

model = myCNN.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

num_epochs = 100
best_val_loss = float("inf")
patience = 10
epochs_no_improve = 0
best_state = None
best_val_metrics = None
best_epoch = None
history_dict = {"train_loss": [], "val_loss": [], "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": []}

for epoch in len(num_epochs):
    
    train_loss = train_for_one_epoch(model, train_loader, device, aug, optimizer, criterion)
    val_loss, val_metrics = val_for_one_epoch(model, val_loader, metrics, device, criterion)

    history_dict["train_loss"].append(train_loss)
    history_dict["val_loss"].append(val_loss)
    history_dict["val_accuracy"].append(val_metrics["accuracy"])
    history_dict["val_f1"].append(val_metrics["f1"])
    history_dict["val_precision"].append(val_metrics["precision"])
    history_dict["val_recall"].append(val_metrics["recall"])

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/100 | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} "
          f"| val_f1: {val_metrics['f1']:.4f} | val_acc: {val_metrics['accuracy']:.4f}")
    
    plot_and_save_loss(train_loss, val_loss, loss_curve_fp)

    # Early stopping

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_metrics = metrics
        best_epoch = epoch
        epochs_no_improve = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve +=1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epcoch: {epoch+1}")
            break

model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch+1} | val_loss: {best_val_loss:.4f} | val_metrics: {best_val_metrics}")

    
